# miniLLM 49M — ton modèle publié contre les meilleurs SFT de l'ablation

Un seul tableau, mêmes mesures, mêmes questions :

- **PUB** : le modèle actuellement sur Hugging Face (téléchargé ici, rien n'est modifié sur le dépôt)
- **C_final** : le candidat retenu · **B_final** et **A_final** : repères

Durée : environ 10 min sur T4 (aucun entraînement). **Exécution → type d'exécution → GPU**, puis **Tout exécuter**.

⚠️ La colonne `qa` est à lire avec prudence pour **PUB** : son jeu d'entraînement n'est pas celui de l'ablation, donc il a pu voir certains exemples de la validation (ça ne peut qu'avantager PUB). `persona`, `basics` et `top5g` ne dépendent pas de ce biais.

In [ ]:
# ── 1) Drive, code ──
import os, sys, glob, shutil, json
from google.colab import drive
drive.mount("/content/drive")

cands = (["/content/drive/MyDrive/MiniLLM_v2"] + glob.glob("/content/drive/MyDrive/*/MiniLLM_v2")
         + glob.glob("/content/drive/Shareddrives/*/MiniLLM_v2"))
BASE = next((c for c in cands if os.path.isdir(f"{c}/data")), None)
if BASE is None:
    raise SystemExit("Dossier MiniLLM_v2 introuvable (Drive > Partagés > appui long > Ajouter un raccourci > Mon Drive).")
DATA, CKPT = f"{BASE}/data", f"{BASE}/checkpoints"
CODE_DIR = "/content/minillm_v2"

!pip -q install tokenizers datasets huggingface_hub
if os.path.exists(f"{CODE_DIR}/.git"):
    !git -C {CODE_DIR} fetch -q origin v2.1.1 && git -C {CODE_DIR} checkout -q v2.1.1 && git -C {CODE_DIR} pull -q --ff-only
else:
    !git clone -q -b v2.1.1 https://github.com/bono-p/minillm_v2.git {CODE_DIR}
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
for n in "ABC":
    print(f"sft_{n}/final.pt :", "✓" if os.path.exists(f"{CKPT}/sft_{n}/final.pt") else "✗ MANQUANT")


In [ ]:
# ── 2) Télécharge le modèle publié (checkpoint + tokenizer) ──
from huggingface_hub import hf_hub_download
REPO = "bonopassale/miniLLM_v2.1-49M-42500it_2.79Btoks_0.9ep_20261001"
PUB = "/content/pub"
os.makedirs(PUB, exist_ok=True)
try:
    for f in ("best.pt", "tokenizer.json"):
        hf_hub_download(REPO, f, local_dir=PUB)
except Exception as e:
    raise SystemExit(f"Téléchargement impossible ({e}). Place best.pt et tokenizer.json du dépôt dans {PUB} à la main, puis relance la cellule 3.")
print({f: f"{os.path.getsize(f'{PUB}/{f}') / 1e6:.0f} Mo" for f in ("best.pt", "tokenizer.json")})


In [ ]:
# ── 3) Tableau comparatif (même jeu de validation que l'ablation : celui de A) ──
specs = [f"PUB={PUB}/best.pt"] + [f"{n}_final={CKPT}/sft_{n}/final.pt" for n in "CBA"]
SPECS = " ".join("--ckpt " + s for s in specs)
os.makedirs(f"{BASE}/evals", exist_ok=True)
!python compare_sft.py {SPECS} --val_dir {DATA}/sft_A --out_dir {BASE}/evals --qa_n 300 --tokenizer {DATA}/tokenizer.json


**À m'envoyer :** le tableau et les lignes « Phrase la plus reprise ».
Les réponses aux 337 questions et les connaissances de base ratées de chaque modèle sont dans `MiniLLM_v2/evals/` (`openqa_<nom>.txt`, `basics_<nom>.txt`).